# VSD Lower-Limb Dataset Exploration

Exploratory data analysis of the VSDFullBody lower-limb CT dataset before any preprocessing.

**Objectives:**
1. Catalogue all available subjects and extract DICOM metadata
2. Visualize anatomy coverage (axial, coronal, sagittal views)
3. Analyze HU intensity distributions per subject
4. Identify knee region z-positions via bone cross-section profiling
5. Assess crop margin sufficiency (+/-100mm)
6. Identify subjects to exclude (insufficient anatomy coverage)
7. Cross-dataset comparison with Ruikar fractured dataset

**Input**: Raw DICOM files from `data/raw/VSD_Dataset/`  
**Output**: Exploration figures saved to `reports/figures/vsd_exploration/`


## Cell 1 - Imports & Configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks

# ============================================================
# Configuration
# ============================================================
PROJECT_ROOT = Path(r"c:\Users\Chan Zheng Shao\OneDrive\Desktop\Github Repo\TestProject\TestProject")
VSD_ROOT = PROJECT_ROOT / "data" / "raw" / "VSD_Dataset"
FRACTURED_ROOT = PROJECT_ROOT / "data" / "raw" / "fractured"
FIG_DIR = PROJECT_ROOT / "reports" / "figures" / "vsd_exploration"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Bone detection threshold for cross-section profiling
BONE_HU_THRESHOLD = 200

print(f"VSD root     : {VSD_ROOT}")
print(f"Fractured root: {FRACTURED_ROOT}")
print(f"Figures dir  : {FIG_DIR}")


## Cell 2 - Data Discovery & Metadata

Scan all VSD subject directories, load each DICOM series, and extract metadata:
- Image dimensions, pixel spacing, slice thickness
- Physical z-extent (total scan length in mm)
- Subject demographics from folder naming convention

**Note**: We include ALL subjects here (including 010) to document why it is excluded.

In [ ]:
def discover_all_vsd_cases(vsd_root):
    """Discover ALL VSD subjects and extract metadata (no exclusions)."""
    cases = []
    for subject_name in sorted(os.listdir(vsd_root)):
        subject_dir = vsd_root / subject_name
        if not subject_dir.is_dir():
            continue

        subdirs = [d for d in subject_dir.iterdir() if d.is_dir()]
        if not subdirs:
            continue

        smir_dir = subdirs[0]
        smir_name = smir_dir.name

        # Parse demographics: SMIR.Lower_limb.{Age}Y.{Sex}.CT.{ID}
        parts = smir_name.split(".")
        age = parts[2] if len(parts) > 2 else "?"
        sex = parts[3] if len(parts) > 3 else "?"

        reader = sitk.ImageSeriesReader()
        dicom_names = reader.GetGDCMSeriesFileNames(str(smir_dir))
        if not dicom_names:
            continue
        reader.SetFileNames(dicom_names)
        img = reader.Execute()
        arr = sitk.GetArrayFromImage(img).astype(np.float32)
        sp = img.GetSpacing()

        cases.append({
            "subject_id": subject_name,
            "smir_name": smir_name,
            "age": age,
            "sex": sex,
            "dicom_dir": str(smir_dir),
            "size_xyz": img.GetSize(),
            "spacing_x": round(sp[0], 6),
            "spacing_y": round(sp[1], 6),
            "spacing_z": round(sp[2], 6),
            "n_slices": img.GetSize()[2],
            "phys_z_mm": round(img.GetSize()[2] * sp[2], 1),
            "phys_x_mm": round(img.GetSize()[0] * sp[0], 1),
            "phys_y_mm": round(img.GetSize()[1] * sp[1], 1),
            "hu_min": float(arr.min()),
            "hu_max": float(arr.max()),
            "hu_mean": float(arr.mean()),
            "img": img,
            "arr": arr,
        })
        print(f"  {subject_name} ({age}.{sex}): {img.GetSize()}, "
              f"spacing=({sp[0]:.4f}, {sp[1]:.4f}, {sp[2]:.4f})mm, "
              f"z-extent={img.GetSize()[2] * sp[2]:.0f}mm")

    return cases


print("Scanning VSD dataset (all subjects)...")
print()
all_cases = discover_all_vsd_cases(VSD_ROOT)
print()
print(f"Total subjects found: {len(all_cases)}")

# Summary table
df_meta = pd.DataFrame([{k: v for k, v in c.items() if k not in ('img', 'arr', 'dicom_dir', 'smir_name')}
                         for c in all_cases])
display(df_meta)


## Cell 3 - Per-Subject Overview (4-Panel Visualization)

For each subject, show:
1. **Axial** slice at the volume midpoint
2. **Coronal** slice at the volume midpoint
3. **Sagittal** slice at the volume midpoint
4. **Info panel** with key metadata

This gives a quick visual assessment of anatomy coverage and scan quality.

In [ ]:
for case in all_cases:
    sid = case["subject_id"]
    arr = case["arr"]
    mid_z = arr.shape[0] // 2
    mid_y = arr.shape[1] // 2
    mid_x = arr.shape[2] // 2

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    # Axial
    axes[0].imshow(arr[mid_z], cmap="gray", vmin=-500, vmax=1500)
    axes[0].set_title(f"Axial (z={mid_z})")
    axes[0].axis("off")

    # Coronal
    axes[1].imshow(arr[:, mid_y, :], cmap="gray", vmin=-500, vmax=1500, aspect="auto")
    axes[1].set_title(f"Coronal (y={mid_y})")
    axes[1].axis("off")

    # Sagittal
    axes[2].imshow(arr[:, :, mid_x], cmap="gray", vmin=-500, vmax=1500, aspect="auto")
    axes[2].set_title(f"Sagittal (x={mid_x})")
    axes[2].axis("off")

    # Info panel
    axes[3].axis("off")
    info_text = (
        f"Subject: {sid}\n"
        f"Age/Sex: {case['age']}/{case['sex']}\n"
        f"Size: {case['size_xyz']}\n"
        f"Spacing: ({case['spacing_x']:.4f}, {case['spacing_y']:.4f}, {case['spacing_z']:.4f})\n"
        f"Z-extent: {case['phys_z_mm']:.0f}mm\n"
        f"HU range: [{case['hu_min']:.0f}, {case['hu_max']:.0f}]\n"
        f"HU mean: {case['hu_mean']:.0f}"
    )
    axes[3].text(0.1, 0.5, info_text, transform=axes[3].transAxes,
                fontsize=12, verticalalignment="center", fontfamily="monospace",
                bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))
    axes[3].set_title("Metadata")

    plt.suptitle(f"VSD Subject {sid} - Overview", fontsize=14)
    plt.tight_layout()
    fig_path = FIG_DIR / f"vsd_{sid}_exploration.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fig_path}")


## Cell 4 - HU Intensity Distributions

Plot HU histograms for each subject to understand:
- Where the bone, soft tissue, and air peaks are
- Whether the confirmed bone window [-450, 1050] captures the relevant range
- Any outliers or artifacts in the data

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

for i, case in enumerate(all_cases):
    sid = case["subject_id"]
    arr = case["arr"]

    # Exclude air background (HU < -900) for cleaner histogram
    tissue = arr[arr > -900].ravel()

    axes[i].hist(tissue, bins=200, color="steelblue", alpha=0.7, density=True)
    axes[i].axvline(x=-450, color="red", linestyle="--", alpha=0.7, label="Window low (-450)")
    axes[i].axvline(x=1050, color="red", linestyle="--", alpha=0.7, label="Window high (1050)")
    axes[i].axvline(x=200, color="orange", linestyle=":", alpha=0.7, label="Bone threshold (200)")
    axes[i].set_title(f"{sid} ({case['age']}/{case['sex']})")
    axes[i].set_xlabel("HU")
    axes[i].set_ylabel("Density")
    axes[i].set_xlim(-500, 2000)
    if i == 0:
        axes[i].legend(fontsize=8)

# Hide unused subplot
if len(all_cases) < len(axes):
    for j in range(len(all_cases), len(axes)):
        axes[j].axis("off")

plt.suptitle("HU Intensity Distributions (air excluded, HU > -900)", fontsize=14)
plt.tight_layout()
fig_path = FIG_DIR / "vsd_hu_histograms.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

# Also save individual histograms
for case in all_cases:
    sid = case["subject_id"]
    arr = case["arr"]
    tissue = arr[arr > -900].ravel()

    fig2, ax2 = plt.subplots(figsize=(8, 4))
    ax2.hist(tissue, bins=300, color="steelblue", alpha=0.7, density=True)
    ax2.axvline(x=-450, color="red", linestyle="--", label="Window low (-450)")
    ax2.axvline(x=1050, color="red", linestyle="--", label="Window high (1050)")
    ax2.axvline(x=200, color="orange", linestyle=":", label="Bone threshold (200)")
    ax2.set_title(f"Subject {sid} - HU Distribution")
    ax2.set_xlabel("HU")
    ax2.set_ylabel("Density")
    ax2.set_xlim(-600, 2500)
    ax2.legend()
    plt.tight_layout()
    fig2.savefig(FIG_DIR / f"vsd_{sid}_hu_histogram.png", dpi=150, bbox_inches="tight")
    plt.close(fig2)

print(f"Individual histograms saved for {len(all_cases)} subjects.")


## Cell 5 - Bone Cross-Section Profile & Knee Identification

For each subject, compute the bone cross-section area at each z-slice (number of voxels with HU > 200).
The knee joint creates a distinctive peak because the femoral condyles + tibial plateau form the
widest bone cross-section between hip and ankle.

We smooth the profile and detect peaks to automatically identify the knee z-position.

In [ ]:
def find_knee_center(arr, spacing_z, smooth_sigma=20):
    """Identify knee center by bone cross-section area profile."""
    bone_mask = arr > BONE_HU_THRESHOLD
    bone_area = bone_mask.sum(axis=(1, 2)).astype(float)
    smoothed = gaussian_filter1d(bone_area, sigma=smooth_sigma)

    peaks, props = find_peaks(
        smoothed,
        height=np.max(smoothed) * 0.2,
        distance=80,
        prominence=np.max(smoothed) * 0.05,
    )

    z_mm = np.arange(arr.shape[0]) * spacing_z
    total_z = z_mm[-1]

    # Knee is typically at 40-65% of the lower limb z-extent
    knee_range = (total_z * 0.40, total_z * 0.65)
    candidates = [p for p in peaks if knee_range[0] <= z_mm[p] <= knee_range[1]]

    if candidates:
        knee_slice = max(candidates, key=lambda p: smoothed[p])
    elif len(peaks) > 0:
        mid_z = total_z * 0.5
        knee_slice = peaks[np.argmin(np.abs(z_mm[peaks] - mid_z))]
    else:
        knee_slice = arr.shape[0] // 2  # fallback

    return knee_slice, z_mm, smoothed, bone_area, peaks


# Compute for all subjects
fig, axes = plt.subplots(len(all_cases), 1, figsize=(14, 4 * len(all_cases)))
if len(all_cases) == 1:
    axes = [axes]

knee_positions = []

for i, case in enumerate(all_cases):
    sid = case["subject_id"]
    arr = case["arr"]
    sp_z = case["spacing_z"]

    knee_slice, z_mm, smoothed, raw_area, all_peaks = find_knee_center(arr, sp_z)
    knee_z = z_mm[knee_slice]

    # Store results
    case["knee_slice"] = knee_slice
    case["knee_z_mm"] = round(knee_z, 1)
    case["all_peaks"] = all_peaks
    case["z_mm"] = z_mm
    case["smoothed_profile"] = smoothed

    knee_positions.append({
        "subject_id": sid,
        "knee_slice": knee_slice,
        "knee_z_mm": round(knee_z, 1),
        "total_z_mm": round(z_mm[-1], 1),
        "knee_pct": round(knee_z / z_mm[-1] * 100, 1),
        "n_peaks": len(all_peaks),
    })

    # Plot
    axes[i].plot(z_mm, raw_area, alpha=0.3, color="gray", label="Raw bone area")
    axes[i].plot(z_mm, smoothed, color="steelblue", linewidth=2, label="Smoothed")
    for p in all_peaks:
        axes[i].axvline(x=z_mm[p], color="lightcoral", alpha=0.5, linestyle=":")
    axes[i].axvline(x=knee_z, color="red", linewidth=2, label=f"Knee: {knee_z:.0f}mm")

    # Show crop margin
    axes[i].axvspan(knee_z - 100, knee_z + 100, alpha=0.15, color="green",
                    label="Crop margin (+/-100mm)")

    axes[i].set_title(f"Subject {sid} - Bone Cross-Section Profile")
    axes[i].set_xlabel("Z position (mm)")
    axes[i].set_ylabel("Bone area (voxels)")
    axes[i].legend(loc="upper right", fontsize=8)

plt.suptitle("Bone Cross-Section Area Profiles & Knee Identification", fontsize=14, y=1.01)
plt.tight_layout()
fig_path = FIG_DIR / "vsd_bone_profiles.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

# Print knee position summary
df_knee = pd.DataFrame(knee_positions)
display(df_knee)


## Cell 5b - Subject Inclusion Screening

**How to decide if a subject should be included or excluded.**

Two automatic checks are applied:
1. **Z-extent**: scans shorter than 700mm almost certainly do not contain the full knee region
2. **Knee peak detected**: the bone cross-section profile must have a detectable peak in the 40-65% z-range

The **full coronal view** is the most powerful visual check — it shows the entire anatomy
from ankle to hip in a single image, so you can immediately see whether the knee joint is
present. A knee-containing scan shows a clear joint space (femoral condyles + tibial plateau)
in the middle of the image.

> **For new subjects**: run this cell after adding them to `data/raw/VSD_Dataset/`.
> Red-flagged subjects should be inspected manually before deciding to exclude.

In [ ]:
# ============================================================
# Automatic inclusion screening
# ============================================================
MIN_Z_MM = 700  # scans shorter than this almost certainly miss the knee

screening_results = []
for case in all_cases:
    sid = case["subject_id"]
    arr = case["arr"]
    sp_z = case["spacing_z"]
    z_mm_arr = np.arange(arr.shape[0]) * sp_z
    total_z = z_mm_arr[-1]

    # Check 1: is the scan long enough?
    long_enough = total_z >= MIN_Z_MM

    # Check 2: is there a knee peak in the expected z-range?
    bone_area = (arr > BONE_HU_THRESHOLD).sum(axis=(1, 2)).astype(float)
    smoothed = gaussian_filter1d(bone_area, sigma=20)
    from scipy.signal import find_peaks
    peaks, _ = find_peaks(smoothed, height=smoothed.max() * 0.2,
                          distance=80, prominence=smoothed.max() * 0.05)
    knee_range = (total_z * 0.40, total_z * 0.65)
    knee_candidates = [p for p in peaks if knee_range[0] <= z_mm_arr[p] <= knee_range[1]]
    has_knee_peak = len(knee_candidates) > 0

    include = long_enough and has_knee_peak
    status = "INCLUDE" if include else "EXCLUDE"

    screening_results.append({
        "subject_id": sid,
        "phys_z_mm": round(total_z, 0),
        "long_enough (>=700mm)": long_enough,
        "knee_peak_detected": has_knee_peak,
        "n_knee_candidates": len(knee_candidates),
        "recommendation": status,
    })

df_screen = pd.DataFrame(screening_results)
print("Subject Inclusion Screening Results")
print("=" * 50)
display(df_screen)

excluded = [r for r in screening_results if r["recommendation"] == "EXCLUDE"]
included = [r for r in screening_results if r["recommendation"] == "INCLUDE"]
print(f"\nINCLUDE: {[r['subject_id'] for r in included]}")
print(f"EXCLUDE: {[r['subject_id'] for r in excluded]}")

# ============================================================
# Full coronal view for every subject
# This is the most intuitive visual check - one image per subject
# showing the entire anatomy from ankle to hip.
# A knee-containing scan has a visible joint space at mid-image.
# ============================================================
n_cases = len(all_cases)
fig, axes = plt.subplots(1, n_cases, figsize=(5 * n_cases, 14))
if n_cases == 1:
    axes = [axes]

for ax, case, result in zip(axes, all_cases, screening_results):
    sid = case["subject_id"]
    arr = case["arr"]
    mid_y = arr.shape[1] // 2

    # Coronal slice spans full z-extent
    coronal = arr[:, mid_y, :]  # shape: (z, x)
    ax.imshow(coronal, cmap="gray", vmin=-500, vmax=1500, aspect="auto",
              extent=[0, arr.shape[2], arr.shape[0], 0])

    # Mark the expected knee zone (40-65% z-range)
    z_total = arr.shape[0]
    zone_lo = int(z_total * 0.40)
    zone_hi = int(z_total * 0.65)
    ax.axhspan(zone_lo, zone_hi, alpha=0.15, color="yellow", label="Expected knee zone")

    # Draw the detected knee center if available
    if "knee_slice" in case:
        ax.axhline(y=case["knee_slice"], color="red", linewidth=2,
                   label=f"Knee: {case['knee_z_mm']:.0f}mm")

    # Title colored by recommendation
    rec = result["recommendation"]
    color = "green" if rec == "INCLUDE" else "red"
    title = f"{sid}\n{rec}\nZ={result['phys_z_mm']:.0f}mm"
    ax.set_title(title, color=color, fontweight="bold", fontsize=12)
    ax.set_xlabel("X (pixel)")
    ax.set_ylabel("Z slice (ankle -> hip)")

    # Add legend only for first subject
    if sid == all_cases[0]["subject_id"]:
        ax.legend(loc="upper right", fontsize=7)

plt.suptitle(
    "Full Coronal Views - Subject Inclusion Screening\n"
    "(Yellow band = expected knee zone, Red line = detected knee, "
    "Green title = INCLUDE, Red title = EXCLUDE)",
    fontsize=13, y=1.01
)
plt.tight_layout()
fig_path = FIG_DIR / "subject_screening_coronal.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")
print()
print("HOW TO READ THIS CHART:")
print("  - Yellow band: the z-range (40-65%%) where a knee peak is expected")
print("  - Red line: the automatically detected knee centre")
print("  - If the scan is too short, the yellow band will be in the wrong anatomy")
print("  - Valid subjects show a clear joint space (dark gap between bones) at the red line")
print("  - Excluded subjects either have no visible joint, or the joint is outside the band")


## Cell 6 - Axial Knee Slices (9-Slice Grid)

Visualize 9 axial slices spanning the knee region for each subject.
This verifies that the identified knee center is correct and the
+/-100mm crop margin captures the full joint anatomy
(distal femur, proximal tibia/fibula, patella).

In [ ]:
for case in all_cases:
    sid = case["subject_id"]
    arr = case["arr"]
    ks = case["knee_slice"]
    sp_z = case["spacing_z"]

    # 9 slices spanning +/-80mm around knee
    offsets_mm = np.linspace(-80, 80, 9)
    offsets_slices = (offsets_mm / sp_z).astype(int)
    slice_indices = np.clip(ks + offsets_slices, 0, arr.shape[0] - 1)

    fig, axes = plt.subplots(3, 3, figsize=(12, 12))
    axes_flat = axes.ravel()

    for j, (sl_idx, offset_mm) in enumerate(zip(slice_indices, offsets_mm)):
        axes_flat[j].imshow(arr[sl_idx], cmap="gray", vmin=-500, vmax=1500)
        # Highlight center slice
        if abs(offset_mm) < 1:
            for spine in axes_flat[j].spines.values():
                spine.set_edgecolor("red")
                spine.set_linewidth(3)
            axes_flat[j].set_title(f"z={offset_mm:+.0f}mm (KNEE CENTER)", color="red", fontweight="bold")
        else:
            axes_flat[j].set_title(f"z={offset_mm:+.0f}mm (slice {sl_idx})")
        axes_flat[j].axis("off")

    plt.suptitle(f"Subject {sid} - 9 Axial Slices Through Knee Region", fontsize=14)
    plt.tight_layout()
    fig_path = FIG_DIR / f"vsd_{sid}_knee_axial.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fig_path}")


## Cell 7 - Subject 010 Analysis & Exclusion

Subject 010 has a notably shorter z-extent. Investigate whether it
actually contains the knee region or should be excluded from processing.

In [ ]:
# Find subject 010
case_010 = next((c for c in all_cases if c["subject_id"] == "010"), None)

if case_010 is not None:
    arr_010 = case_010["arr"]
    sp_z = case_010["spacing_z"]
    z_mm = np.arange(arr_010.shape[0]) * sp_z

    print("Subject 010 Analysis:")
    print(f"  Shape: {arr_010.shape}")
    print(f"  Z-extent: {z_mm[-1]:.0f}mm")
    print(f"  Spacing: ({case_010['spacing_x']:.4f}, {case_010['spacing_y']:.4f}, {sp_z:.4f})mm")
    print()

    # Compare with other subjects
    other_z = [c["phys_z_mm"] for c in all_cases if c["subject_id"] != "010"]
    print(f"  Other subjects z-extent: {min(other_z):.0f} - {max(other_z):.0f}mm (mean={np.mean(other_z):.0f}mm)")
    print(f"  Subject 010 z-extent:    {case_010['phys_z_mm']:.0f}mm")
    print(f"  Ratio: {case_010['phys_z_mm'] / np.mean(other_z) * 100:.0f}%% of mean")
    print()

    # Bone profile for 010
    bone_area_010 = (arr_010 > BONE_HU_THRESHOLD).sum(axis=(1, 2)).astype(float)
    smoothed_010 = gaussian_filter1d(bone_area_010, sigma=20)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Bone profile
    axes[0].plot(z_mm, smoothed_010, color="steelblue", linewidth=2)
    axes[0].set_title("010 - Bone Cross-Section Profile")
    axes[0].set_xlabel("Z position (mm)")
    axes[0].set_ylabel("Bone area")

    # Show what a knee position would need to be
    total_z = z_mm[-1]
    expected_knee = total_z * 0.5
    axes[0].axvline(x=expected_knee, color="red", linestyle="--",
                    label=f"Expected knee ~{expected_knee:.0f}mm")
    axes[0].legend()

    # Axial at bottom (ankle end)
    axes[1].imshow(arr_010[0], cmap="gray", vmin=-500, vmax=1500)
    axes[1].set_title("010 - Bottom slice (z=0mm)")
    axes[1].axis("off")

    # Axial at top (hip end)
    axes[2].imshow(arr_010[-1], cmap="gray", vmin=-500, vmax=1500)
    axes[2].set_title(f"010 - Top slice (z={total_z:.0f}mm)")
    axes[2].axis("off")

    plt.suptitle("Subject 010 - Exclusion Analysis", fontsize=14)
    plt.tight_layout()
    fig_path = FIG_DIR / "vsd_010_exclusion.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()

    print(f"CONCLUSION: Subject 010 scan is only {total_z:.0f}mm long.")
    print("It covers mid-thigh to pelvis and does NOT contain the knee region.")
    print("EXCLUDE from all downstream processing.")
else:
    print("Subject 010 not found in dataset.")


## Cell 8 - Cross-Dataset Comparison (VSD Healthy vs Ruikar Fractured)

Compare key characteristics between the VSD healthy dataset and the Ruikar fractured dataset
to understand what standardization work is needed:
- Pixel spacing ranges
- Slice thickness ranges
- Volume dimensions
- Total sample count and split feasibility

In [ ]:
# Load fractured dataset metadata
fractured_cases = []

for part in ["PartLeft", "PartRight"]:
    part_dir = FRACTURED_ROOT / part
    if not part_dir.exists():
        continue
    for case_name in sorted(os.listdir(part_dir)):
        case_dir = part_dir / case_name
        if not case_dir.is_dir():
            continue

        reader = sitk.ImageSeriesReader()
        dicom_names = reader.GetGDCMSeriesFileNames(str(case_dir))
        if not dicom_names:
            continue
        reader.SetFileNames(dicom_names)
        img = reader.Execute()
        sp = img.GetSpacing()
        sz = img.GetSize()

        fractured_cases.append({
            "case_id": case_name,
            "part": part,
            "size_xyz": sz,
            "spacing_x": round(sp[0], 6),
            "spacing_y": round(sp[1], 6),
            "spacing_z": round(sp[2], 6),
            "n_slices": sz[2],
            "phys_z_mm": round(sz[2] * sp[2], 1),
        })

print(f"Fractured cases loaded: {len(fractured_cases)}")
df_frac = pd.DataFrame(fractured_cases)
display(df_frac)


In [ ]:
# Exclude 010 from VSD for comparison
vsd_valid = [c for c in all_cases if c["subject_id"] != "010"]

# Gather comparison metrics
vsd_sp_xy = [c["spacing_x"] for c in vsd_valid]
vsd_sp_z = [c["spacing_z"] for c in vsd_valid]
vsd_z_mm = [c["phys_z_mm"] for c in vsd_valid]
vsd_nslices = [c["n_slices"] for c in vsd_valid]

frac_sp_xy = [c["spacing_x"] for c in fractured_cases]
frac_sp_z = [c["spacing_z"] for c in fractured_cases]
frac_z_mm = [c["phys_z_mm"] for c in fractured_cases]
frac_nslices = [c["n_slices"] for c in fractured_cases]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. XY spacing comparison
axes[0, 0].hist(vsd_sp_xy, bins=10, alpha=0.7, color="steelblue", label="VSD (healthy)")
axes[0, 0].hist(frac_sp_xy, bins=10, alpha=0.7, color="coral", label="Ruikar (fractured)")
axes[0, 0].set_title("XY Pixel Spacing (mm)")
axes[0, 0].set_xlabel("Spacing (mm)")
axes[0, 0].legend()

# 2. Z spacing comparison
axes[0, 1].hist(vsd_sp_z, bins=10, alpha=0.7, color="steelblue", label="VSD")
axes[0, 1].hist(frac_sp_z, bins=10, alpha=0.7, color="coral", label="Ruikar")
axes[0, 1].set_title("Z Spacing / Slice Thickness (mm)")
axes[0, 1].set_xlabel("Spacing (mm)")
axes[0, 1].legend()

# 3. Number of slices
axes[0, 2].bar(["VSD"], [np.mean(vsd_nslices)], yerr=[np.std(vsd_nslices)],
               color="steelblue", alpha=0.7, capsize=5)
axes[0, 2].bar(["Ruikar"], [np.mean(frac_nslices)], yerr=[np.std(frac_nslices)],
               color="coral", alpha=0.7, capsize=5)
axes[0, 2].set_title("Number of Slices (mean +/- std)")
axes[0, 2].set_ylabel("Slices")

# 4. Z-extent
axes[1, 0].bar(["VSD"], [np.mean(vsd_z_mm)], yerr=[np.std(vsd_z_mm)],
               color="steelblue", alpha=0.7, capsize=5)
axes[1, 0].bar(["Ruikar"], [np.mean(frac_z_mm)], yerr=[np.std(frac_z_mm)],
               color="coral", alpha=0.7, capsize=5)
axes[1, 0].set_title("Physical Z-Extent (mm, mean +/- std)")
axes[1, 0].set_ylabel("mm")

# 5. Sample counts
vsd_knees = len(vsd_valid) * 2  # bilateral
frac_knees = len(fractured_cases)
axes[1, 1].bar(["VSD\n(healthy)", "Ruikar\n(fractured)"], [vsd_knees, frac_knees],
               color=["steelblue", "coral"], alpha=0.7)
axes[1, 1].set_title("Total Knee Volumes")
axes[1, 1].set_ylabel("Count")
for j, v in enumerate([vsd_knees, frac_knees]):
    axes[1, 1].text(j, v + 0.3, str(v), ha="center", fontweight="bold")

# 6. Summary table
axes[1, 2].axis("off")
summary = (
    "CROSS-DATASET SUMMARY\n"
    "=====================\n"
    f"VSD (healthy):     {len(vsd_valid)} subjects, {vsd_knees} knees\n"
    f"Ruikar (fractured): {len(fractured_cases)} cases\n"
    f"Total volumes:      {vsd_knees + frac_knees}\n"
    f"\nVSD XY spacing:  {min(vsd_sp_xy):.3f}-{max(vsd_sp_xy):.3f}mm\n"
    f"Ruikar XY spacing: {min(frac_sp_xy):.3f}-{max(frac_sp_xy):.3f}mm\n"
    f"VSD Z spacing:   {min(vsd_sp_z):.3f}-{max(vsd_sp_z):.3f}mm\n"
    f"Ruikar Z spacing:  {min(frac_sp_z):.3f}-{max(frac_sp_z):.3f}mm\n"
    f"\nStandardization: resample to 0.5mm isotropic"
)
axes[1, 2].text(0.05, 0.5, summary, transform=axes[1, 2].transAxes,
                fontsize=10, verticalalignment="center", fontfamily="monospace",
                bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))

plt.suptitle("Cross-Dataset Comparison: VSD Healthy vs Ruikar Fractured", fontsize=14)
plt.tight_layout()
fig_path = FIG_DIR / "cross_dataset_comparison.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")


## Exploration Findings

### VSD Dataset Summary
- **6 subjects** downloaded (001, 002, 005, 006, 010, 014)
- **Subject 010 excluded** - scan only ~598mm, covers mid-thigh to pelvis, does not contain knee region
- **5 valid subjects**, all bilateral (both legs) = **10 healthy knee volumes**
- All scans are 512x512 axial with non-uniform z-spacing (handled by resampling)

### Knee Identification
- Knee z-positions: 001=578mm, 002=518mm, 005=566mm, 006=567mm, 014=523mm
- Bone cross-section profiling reliably identifies the knee in the 40-65% z-range
- Crop margin of +/-100mm is sufficient to capture the full joint

### Cross-Dataset Comparison
- VSD: lower limb full-length scans, ~0.6mm z-spacing, ~0.86-0.98mm xy-spacing
- Ruikar: knee-region DICOM, variable z-spacing (0.7mm or 3.0mm), ~0.37-0.55mm xy-spacing
- Both datasets require resampling to 0.5mm isotropic for standardization
- Total volumes: 10 healthy + 14 fractured = **24 volumes** (small, plan augmentation)

### Next Steps
1. Run `vsd_knee_cropping.ipynb` to crop and separate bilateral legs
2. Build unified preprocessing notebook (resample, orient, window, ROI crop, resize)
3. Generate DRRs with DiffDRR and compare healthy vs fractured representations